In [ ]:
import mne
import numpy as np
import pandas as pd
import os
from scipy.signal import welch
from scipy.integrate import trapezoid

data_path = r"C:\Users\hiro2\Documents\EEG_project\data\MUSIN-G"

genre_map = {
    1: "Deep House", 2: "Indie", 3: "Electronics",
    4: "New Age", 5: "Electronic Dance", 6: "Ambient",
    7: "Hindustani Classical", 8: "Indian Semi-Classical",
    9: "Indian Folk", 10: "Soft Jazz", 11: "Goth Rock",
    12: "Progressive Instrumental"
}

# 行動データ読み込み（区切り文字を自動判定）
behav_path = os.path.join(data_path, "stimuli", "Behavioural_data")
behav_df = pd.read_csv(behav_path, sep='\s+', engine='python')
behav_df.columns = behav_df.columns.str.strip()
print("カラム名:", behav_df.columns.tolist())
print("行動データ先頭:\n", behav_df.head())

# sub-001のEnjoyment確認
sub1 = behav_df[behav_df['Subject'] == 1]
print(f"\nsub-001のEnjoyment評価: {sub1['Enjoyment'].values}")

# EEG読み込み・前処理
set_path = os.path.join(data_path, "sub-001", "ses-01", "eeg",
                        "sub-001_ses-01_task-MusicListening_run-1_eeg.set")
raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
raw.filter(1., 40., fir_window='hamming', verbose=False)

# バンドパワー計算
def band_power(data, sf, band):
    freqs, psd = welch(data, fs=sf, nperseg=int(sf*2))
    idx = np.logical_and(freqs >= band[0], freqs <= band[1])
    return trapezoid(psd[:, idx], freqs[idx], axis=1).mean()

sf = raw.info['sfreq']
data = raw.get_data()

theta = band_power(data, sf, [4, 8])
alpha = band_power(data, sf, [8, 13])
beta  = band_power(data, sf, [13, 30])

print(f"\n=== バンドパワー（sub-001, ses-01）===")
print(f"Theta: {theta:.4e}")
print(f"Alpha: {alpha:.4e}")
print(f"Beta:  {beta:.4e}")
print(f"Beta/Alpha比 (Arousal): {beta/alpha:.4f}")